# HDFS ablation campaign — Google Colab

Train and evaluate a **prepared** HDFS campaign. Parsing, LLM enrichment, and graph construction stay local; this notebook only stages `.pt.gz` bundles and runs GAE training.

**Families**

- **A (representation)** — one rebuilt graph per arm (`tfidf_only`, `no_llm_enrichment`, …). Stage **one** gzip at a time.
- **B (train)** — reuse `baseline_full` and sweep architecture/loss (`alpha_0`, `gine_mean_agg`, …).

See `notebooks/ABLATION.md`. Do not pass `configs/ablation_representation.yaml` with a single shared `--graph-dataset`.

Single-run debugging (no campaign): `6_GAE_Training_Colab.ipynb`.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

In [ ]:
from pathlib import Path
import gzip
import json
import os
import shutil
import subprocess

REPOSITORY_URL = "https://github.com/michaail/hybrid-logs-analyzer-research.git"
GIT_REF = "ablation-cursor"  # pin the campaign branch; do not pull mid-run
DATASET = "hdfs"
CAMPAIGN_ID = "hdfs_ablation_20260916"
FAMILY = "B"                 # "A" (per-graph representation) or "B" (one graph, train matrix)
SMOKE = True                 # 1 epoch / 5k graphs to prove Drive staging
REUSE_DRIVE_CACHE = True

# Family B without a campaign folder: path of one hybrid notebook_raw_v1 graph on Drive.
GRAPH_DATASET_RELATIVE_PATH = (
    f"campaigns/{CAMPAIGN_ID}/graphs/baseline_full/graph_dataset.pt.gz"
)
CAMPAIGN_RELATIVE_DIR = f"campaigns/{CAMPAIGN_ID}"

def find_local_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "src" / "modules" / "models" / "gae.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the hybrid-logs-analyzer-research checkout."
    )

REPO_ROOT = Path("/content/hybrid-logs-analyzer-research") if IN_COLAB else find_local_repository_root()
WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else REPO_ROOT
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    if not REPO_ROOT.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", GIT_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ])
    else:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", GIT_REF])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
    DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

    def stage_from_drive(relative_path: str) -> Path:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        destination = WORKSPACE_ROOT / relative_path
        if not source.exists():
            raise FileNotFoundError(f"Missing Drive artifact: {source}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source.is_dir():
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])
        else:
            shutil.copy2(source, destination)
        return destination

    def stage_tree_from_drive(relative_path: str) -> None:
        source = DRIVE_ARTIFACT_ROOT / relative_path
        if source.exists():
            destination = WORKSPACE_ROOT / relative_path
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.check_call(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"])

    if REUSE_DRIVE_CACHE:
        stage_tree_from_drive(f"artifacts/cache/{DATASET}")
        stage_tree_from_drive("artifacts/runs")
        stage_tree_from_drive(f"outputs/{DATASET}")

    CAMPAIGN_DIR = None
    GRAPH_DATASET_PATH = None
    if FAMILY.upper() == "A":
        CAMPAIGN_DIR = stage_from_drive(CAMPAIGN_RELATIVE_DIR)
    else:
        campaign_source = DRIVE_ARTIFACT_ROOT / CAMPAIGN_RELATIVE_DIR / "manifest.json"
        if campaign_source.exists():
            CAMPAIGN_DIR = stage_from_drive(CAMPAIGN_RELATIVE_DIR)
        else:
            GRAPH_DATASET_PATH = stage_from_drive(GRAPH_DATASET_RELATIVE_PATH)
            if GRAPH_DATASET_PATH.suffix == ".gz":
                unpacked = GRAPH_DATASET_PATH.with_suffix("")
                print(f"Decompressing {GRAPH_DATASET_PATH.name}")
                with gzip.open(GRAPH_DATASET_PATH, "rb") as src, open(unpacked, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
                GRAPH_DATASET_PATH = unpacked
else:
    CAMPAIGN_DIR = (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR) if (WORKSPACE_ROOT / CAMPAIGN_RELATIVE_DIR / "manifest.json").exists() else None
    GRAPH_DATASET_PATH = None

print(f"Code checkout : {REPO_ROOT}")
print(f"Workspace     : {WORKSPACE_ROOT}")
print(f"Campaign dir  : {CAMPAIGN_DIR}")
print(f"Family        : {FAMILY}")

In [ ]:
if IN_COLAB:
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "scripts" / "install_colab.py"),
        "--project-root", str(REPO_ROOT),
    ])
else:
    print("Local environment — use this repo venv / requirements.txt, not requirements-colab.txt.")

In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
os.environ["PIPELINE_WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)

gae_module = REPO_ROOT / "src" / "modules" / "models" / "gae.py"
if not gae_module.exists():
    raise FileNotFoundError(
        f"Missing {gae_module}. This Colab clone of {GIT_REF!r} does not contain "
        "the GAE package. Commit and push src/modules/models/, then "
        "Runtime → Restart session and rerun from the clone cell."
    )

import torch
print(f"Working directory: {Path.cwd()}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} | devices: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## Run the campaign

The CLI writes a uniform eval pack per arm under `outputs/hdfs/<campaign_id>_<name>/` and a leaderboard under `outputs/hdfs/campaigns/<campaign_id>/`. Completed arms are skipped on resume.

In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / "configs" / "ablation_base.yaml"
FAMILY_KEY = FAMILY.upper()
runner_command = [
    sys.executable,
    str(REPO_ROOT / "run_ablation.py"),
    "--mode", "train-only",
    "--config", str(BASE_CONFIG_PATH),
    "--workspace-root", str(WORKSPACE_ROOT),
    "--code-root", str(REPO_ROOT),
    "--campaign-id", CAMPAIGN_ID,
    "--family", FAMILY_KEY,
    "--set", f"experiment.dataset={DATASET}",
]
if SMOKE:
    runner_command.extend([
        "--set", "training.test_run=true",
        "--set", "training.epochs=1",
        "--set", "training.test_samples=5000",
    ])
if CAMPAIGN_DIR is not None:
    runner_command.extend(["--campaign-dir", str(CAMPAIGN_DIR)])
elif FAMILY_KEY == "B":
    if GRAPH_DATASET_PATH is None:
        raise ValueError("Family B needs a campaign dir or GRAPH_DATASET_RELATIVE_PATH")
    runner_command.extend([
        "--matrix", str(REPO_ROOT / "configs" / "ablation_train.yaml"),
        "--graph-dataset", str(GRAPH_DATASET_PATH),
    ])
else:
    raise ValueError("Family A requires campaigns/<id>/ on Drive (manifest.json + graphs/).")
if IN_COLAB:
    runner_command.extend(["--checkpoint-root", str(DRIVE_ARTIFACT_ROOT)])

print(" ".join(str(part) for part in runner_command))

def final_drive_sync() -> None:
    if not IN_COLAB:
        return
    for directory_name in ("artifacts", "models", "outputs", "runs", "campaigns"):
        source = WORKSPACE_ROOT / directory_name
        if source.exists():
            destination = DRIVE_ARTIFACT_ROOT / directory_name
            destination.mkdir(parents=True, exist_ok=True)
            subprocess.run(["rsync", "-a", "--partial", f"{source}/", f"{destination}/"], check=True)

try:
    completed = subprocess.run(runner_command, check=False)
finally:
    final_drive_sync()

if completed.returncode:
    raise RuntimeError(f"Campaign runner failed with exit code {completed.returncode}")
print("Campaign train finished.")

## Leaderboard and comparison plots

Loaded from disk (not notebook RAM) so a later session can replay without retraining.

In [ ]:
import pandas as pd
from IPython.display import Image, display

report_root = DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT
campaign_report = report_root / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID
if not (campaign_report / "leaderboard.json").exists():
    subprocess.check_call([
        sys.executable, str(REPO_ROOT / "run_ablation.py"),
        "--mode", "report",
        "--config", str(BASE_CONFIG_PATH),
        "--workspace-root", str(WORKSPACE_ROOT),
        "--code-root", str(REPO_ROOT),
        "--campaign-id", CAMPAIGN_ID,
        "--set", f"experiment.dataset={DATASET}",
    ])
    if IN_COLAB:
        final_drive_sync()
    campaign_report = WORKSPACE_ROOT / "outputs" / DATASET / "campaigns" / CAMPAIGN_ID

leaderboard = pd.read_csv(campaign_report / "leaderboard.csv")
display(leaderboard)
print((campaign_report / "README.md").read_text())
for figure_name in ("ablation_comparison.png", "loss_curves.png", "component_comparison.png"):
    path = campaign_report / figure_name
    if path.exists():
        display(Image(filename=str(path)))

## Replay a previous campaign

Set `CAMPAIGN_ID` above and run only the leaderboard cell. Per-run packs live in `outputs/hdfs/<campaign_id>_<arm>/figures/`.

In [ ]:
from IPython.display import Image, display

runs_root = (DRIVE_ARTIFACT_ROOT if IN_COLAB else WORKSPACE_ROOT) / "outputs" / DATASET
for run_dir in sorted(runs_root.glob(f"{CAMPAIGN_ID}_*")):
    if not (run_dir / "metrics.json").exists():
        continue
    metrics = json.loads((run_dir / "metrics.json").read_text())
    print(f"{run_dir.name}: F1={metrics.get('test_f1')} PR-AUC={metrics.get('test_pr_auc')} ROC-AUC={metrics.get('test_roc_auc')}")
    preview = run_dir / "figures" / "test_pr_roc.png"
    if preview.exists():
        display(Image(filename=str(preview)))